# 78 — Validate the overlay A-Box against the overlay shapes

The `60_` analogue. The A-Box **does not conform**, and every violation traces to a
decision that chose the graph reading over the document reading — the same
situation `60_validate_instances.ipynb` records for the core under D11, at overlay
scale. The notebook classifies every result and **fails if a violation appears that
is not one of the recorded causes**.

What conforms is the part D24 was written for: after the enum repair, every
`resultScale`, `permissibleValue` and `relation` value satisfies its `sh:in`, and the
three-value admission of D23 holds in the deliverable, not only in a renderer guard.

One shape validates nothing, and that is a D19 measurement rather than an error:
`gen-shacl` honours `class_uri`, so the stand-in's node shape targets
`cosmos_sdtm:AssignedTerm`, while `gen-owl` does not, so the T-Box declares
`qbc:AssignedTerm` and the A-Box types a recording's specimen that way. The eight
specimen nodes therefore fail the recording's `sh:class cosmos_sdtm:AssignedTerm`
(recorded below) and are never reached by the `AssignedTerm` shape itself — its
`conceptId` pattern goes untested. The notebook asserts the count so a generator
change that closes the gap is noticed.

Requires `pyshacl`.

## Configuration

In [ ]:
ROOT    = ".."
REPORTS = "../reports"

INSTANCES = f"{ROOT}/cosmos_qbc_v1.instances.ttl"
SHAPES    = f"{ROOT}/cosmos_qbc_v1.shapes.ttl"

# Every expected (constraint, path) pair, and why it happens.
# A result outside this map fails the notebook.
EXPECTED = {
    ("DatatypeConstraintComponent", "broaderConceptId"):
        "schema range is a C-code string; the A-Box renders the analyte link as an edge to the NCIt PURL (D2, D14)",
    ("NodeKindConstraintComponent", "broaderConceptId"):
        "schema range is a C-code string; the A-Box renders the analyte link as an edge to the NCIt PURL (D2, D14)",
    ("PatternConstraintComponent", "broaderConceptId"):
        "schema range is a C-code string; the A-Box renders the analyte link as an edge to the NCIt PURL (D2, D14)",
    ("DatatypeConstraintComponent", "conceptId"):
        "schema range is a C-code string; the use-node's conceptId is an edge to the NCIt PURL (D2, D22)",
    ("NodeKindConstraintComponent", "conceptId"):
        "schema range is a C-code string; the use-node's conceptId is an edge to the NCIt PURL (D2, D22)",
    ("PatternConstraintComponent", "conceptId"):
        "schema range is a C-code string; the use-node's conceptId is an edge to the NCIt PURL (D2, D22)",
    ("DatatypeConstraintComponent", "stateDataElementConceptId"):
        "schema range is a C-code string; the assertion's state DEC is an edge to the NCIt PURL (D2, D15)",
    ("NodeKindConstraintComponent", "stateDataElementConceptId"):
        "schema range is a C-code string; the assertion's state DEC is an edge to the NCIt PURL (D2, D15)",
    ("PatternConstraintComponent", "stateDataElementConceptId"):
        "schema range is a C-code string; the assertion's state DEC is an edge to the NCIt PURL (D2, D15)",
    ("ClosedConstraintComponent", "broader"):
        "authored: skos:broader carries the analyte link beside the schema's own slot (D14)",
    ("ClosedConstraintComponent", "exactMatch"):
        "authored: the mapping relation is emitted as the SKOS predicate it names (D16, D23)",
    ("ClosedConstraintComponent", "narrowMatch"):
        "authored: the mapping relation is emitted as the SKOS predicate it names (D16, D23)",
    ("ClosedConstraintComponent", "broadMatch"):
        "authored: the mapping relation is emitted as the SKOS predicate it names (D16, D23)",
    ("ClosedConstraintComponent", "identifier"):
        "authored: dcterms:identifier carries the recording mnemonic; the subject is the Dataset Specialization IRI (D17)",
    ("ClosedConstraintComponent", "domain"):
        "D19 divergence: gen-shacl honours slot_uri cosmos_sdtm:domain, gen-owl drops it; the A-Box follows the T-Box and writes qbc:domain",
    ("MinCountConstraintComponent", "domain"):
        "D19 divergence: gen-shacl honours slot_uri cosmos_sdtm:domain, gen-owl drops it; the A-Box follows the T-Box and writes qbc:domain",
    ("ClassConstraintComponent", "specimen"):
        "D19 divergence: gen-shacl honours class_uri cosmos_sdtm:AssignedTerm, gen-owl mints qbc:AssignedTerm; the A-Box follows the T-Box and types the node so",
}

## Validate

In [ ]:
import csv
from collections import Counter
from pathlib import Path

from pyshacl import validate
from rdflib import Graph
from rdflib.namespace import RDF, SH

data = Graph().parse(INSTANCES, format="turtle")
shapes = Graph().parse(SHAPES, format="turtle")

conforms, results, _ = validate(data, shacl_graph=shapes, inference="none", advanced=True)
print(f"conforms: {conforms}")
print(f"results:  {len(list(results.subjects(RDF.type, SH.ValidationResult))):,}")

# D19: the AssignedTerm shape targets the class_uri; no node in the A-Box carries it.
from rdflib import URIRef

unreached = set(data.subjects(RDF.type, URIRef("https://w3id.org/cdisc/cosmos/qbc/AssignedTerm")))
targeted = set(data.subjects(RDF.type, URIRef("https://www.cdisc.org/cosmos/sdtm_v1.0/AssignedTerm")))
print(f"AssignedTerm nodes the shape targets: {len(targeted)}; typed as the T-Box declares, unreached: {len(unreached)}")
if targeted or len(unreached) != 8:
    raise RuntimeError("the D19 shape gap changed; re-read the recorded causes")

## Classify every result

Grouped by the focus node's type as well, because the same slot name means a
different decision on a different class — `conceptId` on a specimen use-node is D22,
on a recording's assigned term it would be D19.

In [ ]:
def local(term):
    if term is None:
        return "-"
    text = str(term)
    return text.rsplit("#", 1)[-1].rsplit("/", 1)[-1]


rows, counts, unexplained = [], Counter(), []

for result in results.subjects(RDF.type, SH.ValidationResult):
    component = local(results.value(result, SH.sourceConstraintComponent))
    path = local(results.value(result, SH.resultPath))
    focus = results.value(result, SH.focusNode)
    focus_type = local(data.value(focus, RDF.type))

    cause = EXPECTED.get((component, path))
    if cause is None:
        unexplained.append((focus_type, component, path, str(focus)))
    else:
        counts[(focus_type, component, path, cause)] += 1

    rows.append({
        "focus_type": focus_type,
        "constraint": component,
        "path": path,
        "focus_node": str(focus),
        "cause": cause or "UNEXPLAINED",
    })

for (focus_type, component, path, cause), n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"{n:>5,}  {focus_type:30s} {component:28s} {path:26s} {cause[:60]}")
print()
print(f"unexplained results: {len(unexplained):,}")
for u in unexplained[:10]:
    print("   ", u)

## Write the conformance report

In [ ]:
out = Path(REPORTS, "qbc_shacl_conformance.csv")
with open(out, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["focus_type", "constraint", "path", "focus_node", "cause"])
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda r: (r["cause"], r["focus_type"], r["constraint"], r["path"], r["focus_node"])))

summary = Path(REPORTS, "qbc_shacl_conformance_summary.csv")
with open(summary, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["focus_type", "constraint", "path", "cause", "count"])
    for (focus_type, component, path, cause), n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
        writer.writerow([focus_type, component, path, cause, n])

print(f"wrote {out}       ({len(rows):,} rows)")
print(f"wrote {summary}   ({len(counts)} (type, constraint, path) groups)")

## Result

Non-conformance is expected and recorded; an unaccounted violation is not. And the
constraints D24 repaired must produce **no** result at all — that is the positive
half, asserted rather than assumed.

In [ ]:
if unexplained:
    raise RuntimeError(
        f"{len(unexplained):,} validation result(s) fall outside the recorded causes. "
        "Read reports/qbc_shacl_conformance.csv before changing anything - an unexplained "
        "violation is either a rendering bug or a new finding, and both matter."
    )

repaired = [r for r in rows if r["constraint"] == "InConstraintComponent"]
if repaired:
    raise RuntimeError(f"D24: {len(repaired)} sh:in result(s) after the enum repair: {repaired[:3]}")

print(f"{len(rows):,} violations, every one traced to a decision.")
print("0 sh:in violations: resultScale, permissibleValue and relation conform to the repaired lists (D24),")
print("and every qualified concept's result scale is one of the three the overlay admits (D23).")
print()
print("The overlay A-Box does not conform to its own generated shapes, by design:")
print("  - three causes are a C-code the schema types as string, rendered as an edge to the NCIt PURL (D2; D14, D15, D22)")
print("  - two are SKOS predicates emitted beside the schema's own slots (D14, D16)")
print("  - one is dcterms:identifier on a recording whose subject is the DSS IRI (D17)")
print("  - two are the SDTM stand-ins, where gen-shacl honours class_uri and slot_uri and gen-owl does not (D19)")